# Example 11 — The Blasius boundary layer

THE boundary-layer equation of fluid dynamics. Prandtl's boundary-layer equations for a
flat plate admit a similarity solution: with $\eta = y\sqrt{U/(\nu x)}$ and streamfunction
$\psi = \sqrt{U\nu x}\,f(\eta)$, the PDE system collapses to one nonlinear ODE:
$$f''' + \tfrac{1}{2}\,f\,f'' = 0,\qquad f(0)=0,\quad f'(0)=0,\quad f'(\infty)=1,$$
where $f'(\eta) = u/U$ is the velocity profile. The famous constant $f''(0) = 0.332057$
gives the flat-plate skin friction $C_f = 0.664/\sqrt{Re_x}$ — one of the most-used numbers
in aerodynamics.

**This notebook is also a lesson in PINN formulation.** The naive approach fails, and each
fix is a technique from earlier examples:
1. **Naive (soft BCs on $f$):** $f\equiv 0$ satisfies the ODE exactly with $f(0)=f'(0)=0$ —
   only the far-field BC objects. The trivial-solution trap of Example 8 again; training
   stalls near zero.
2. **Third-order autograd:** even with hard BCs, differentiating the network three times is
   poorly conditioned — convergence crawls.
3. **What works (used below):** reduce to a **first-order system** $(f, g=f', h=f'')$ with
   three network outputs — exactly how classical ODE solvers handle high-order equations —
   and build the BCs **hard into trial functions** so no impostor solution is reachable:
   $$f = \eta\,N_f,\qquad g = (1-e^{-\eta}) + s(1-s)\,N_g,\qquad h = N_h,\qquad s=\eta/L.$$
   $g$ is *forced* to run from 0 to 1; the trap is gone by construction.

**Reference:** classical RK4 shooting with the known $f''(0)$.

> Colab: Runtime → GPU. Trains in ~30 s on GPU (~2.5 min CPU).

In [ ]:
# Cell 1 -- Reference: RK4 shooting (the classical method)
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

L = 10.0                          # 'infinity' — f' is 1 to 6 digits well before eta=10

def rk4_shooting(fpp0=0.332057, n=4000):
    h = L/n
    def rhs(y): return np.array([y[1], y[2], -0.5*y[0]*y[2]])
    Y = np.zeros((n+1, 3)); y = np.array([0.0, 0.0, fpp0]); Y[0] = y
    for i in range(n):
        k1=rhs(y); k2=rhs(y+h/2*k1); k3=rhs(y+h/2*k2); k4=rhs(y+h*k3)
        y = y + h/6*(k1+2*k2+2*k3+k4); Y[i+1] = y
    return np.linspace(0, L, n+1), Y

eta_r, Yr = rk4_shooting()
print(f"shooting check: f'(10) = {Yr[-1,1]:.6f} (→1),  "
      f"displacement thickness eta* = {10-Yr[-1,0]:.4f} (known: 1.7208)")

In [ ]:
# Cell 2 -- PINN: first-order system + hard-constrained trial functions
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(1, 64), nn.Tanh(),
                    nn.Linear(64, 64), nn.Tanh(),
                    nn.Linear(64, 64), nn.Tanh(),
                    nn.Linear(64, 3)).to(device)   # 3 outputs -> (f, g, h)

def fgh(s):
    """Trial functions with HARD boundary conditions.  s = eta/L in [0,1].
    f(0)=0        via the eta factor
    g(0)=0,g(L)=1 via the (1-exp(-eta)) base + a bump pinned to 0 at both ends
    h             free (no BC on f'')
    """
    eta = s*L
    o = net(s)
    f = eta*o[:, 0:1]
    g = (1 - torch.exp(-eta)) + s*(1-s)*o[:, 1:2]
    h = o[:, 2:3]
    return f, g, h

opt = torch.optim.Adam(net.parameters(), lr=1e-3)
EPOCHS = 8000
t0 = time.perf_counter()
for e in range(EPOCHS):
    if e == 6000:
        for gr in opt.param_groups: gr['lr'] = 2e-4
    opt.zero_grad()
    s = torch.rand(1024, 1, device=device).requires_grad_(True)
    f, g, h = fgh(s)
    # d/d(eta) = (1/L) d/ds
    fs = torch.autograd.grad(f, s, torch.ones_like(f), create_graph=True)[0]/L
    gs = torch.autograd.grad(g, s, torch.ones_like(g), create_graph=True)[0]/L
    hs = torch.autograd.grad(h, s, torch.ones_like(h), create_graph=True)[0]/L
    loss = ((fs - g)**2).mean() \
         + ((gs - h)**2).mean() \
         + ((hs + 0.5*f*h)**2).mean()        # the Blasius equation itself
    loss.backward(); opt.step()
    if e % 2000 == 0: print(f'epoch {e:5d}  loss {loss.item():.2e}')
if device.type == 'cuda': torch.cuda.synchronize()
print(f'\ntraining: {time.perf_counter()-t0:.0f} s')

In [ ]:
# Cell 3 -- The velocity profile and the skin-friction constant
with torch.no_grad():
    se = torch.tensor(eta_r/L, dtype=torch.float32, device=device).reshape(-1, 1)
f, g, h = fgh(se)
f = f.detach().cpu().numpy().ravel(); g = g.detach().cpu().numpy().ravel()
h = h.detach().cpu().numpy().ravel()

fpp0 = h[0]
err_g = np.sqrt(np.mean((g - Yr[:, 1])**2))
eta99 = eta_r[np.argmax(g >= 0.99)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# the classic BL plot: velocity profile with eta on the vertical axis
ax[0].plot(Yr[:, 1], eta_r, 'g', lw=2.4, label='shooting (RK4)')
ax[0].plot(g, eta_r, 'r--', lw=1.6, label='PINN')
ax[0].axhline(eta99, color='gray', ls=':', lw=1)
ax[0].text(0.05, eta99+0.15, f'$\\eta_{{99}}$ = {eta99:.2f}  (known ≈ 5.0)', color='gray')
ax[0].set_xlabel("$u/U = f'(\\eta)$"); ax[0].set_ylabel('$\\eta$'); ax[0].set_ylim(0, 8)
ax[0].set_title('Blasius velocity profile'); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(eta_r, Yr[:, 2], 'g', lw=2.4, label="shooting f''")
ax[1].plot(eta_r, h, 'r--', lw=1.6, label="PINN f''")
ax[1].set_xlabel('$\\eta$'); ax[1].set_ylabel("$f''(\\eta)$"); ax[1].set_xlim(0, 8)
ax[1].set_title(f"Wall shear:  f''(0) = {fpp0:.5f}   (exact 0.33206)")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"f''(0)            : PINN {fpp0:.5f}   exact 0.332057   ({abs(fpp0-0.332057)/0.332057*100:.2f}% err)")
print(f"skin friction Cf  : PINN {2*fpp0:.4f}/sqrt(Re_x)   exact 0.664/sqrt(Re_x)")
print(f"L2 error of f'    : {err_g:.2e}")
print(f"displacement thk. : PINN {L - f[-1]:.4f}   exact 1.7208")

## Observations (for fluid-dynamics notes)

- **The PINN recovers engineering constants, not just curves.** Wall shear
  $f''(0) = 0.332$ to ~0.1%, hence $C_f = 0.664/\sqrt{Re_x}$; boundary-layer thickness
  $\eta_{99} \approx 5.0$ (the classic $\delta \approx 5.0\,x/\sqrt{Re_x}$); displacement
  thickness $1.72$. These are the numbers a fluid-dynamics course cares about.
- **Formulation is everything.** The naive PINN falls into the trivial-solution trap
  (Example 8's pathology: $f\equiv 0$ solves the ODE; only the far-field BC objects), and
  third-order autograd is ill-conditioned. Two classical ideas fix it: **reduce to a
  first-order system** (as every ODE integrator does) and **hard-constrain the BCs** in
  trial functions so the impostor is unreachable. With those, no loss-weight tuning at all —
  the loss is pure residual.
- **Hard BCs are the strongest medicine in the cabinet.** Soft BC weights (used everywhere
  else in this kit) *negotiate* with the optimizer; trial functions *legislate*. When a
  problem has a trivial impostor, legislation wins.
- **Semi-infinite domains are easy for PINNs.** No mesh to truncate and grade — just pick
  $L$ comfortably past the layer and sample. (Compare the meshing care CFD needs at a wall.)
- **vs the classical method:** RK4 shooting is milliseconds and 10 digits — for THIS
  similarity ODE, classical wins (Example 1's honest lesson again). The PINN approach earns
  its keep when there is no similarity reduction: non-similar BLs, pressure gradients,
  suction/blowing — where the same code structure extends and shooting does not.

**Experiments to try:** remove the hard constraints (soft BC weights instead) and watch the
trivial trap re-appear; add wall suction ($f(0) = 0.5$) or blowing ($f(0)=-0.3$) — one-line
changes to the trial function, and watch $f''(0)$ (skin friction) respond; try the
Falkner–Skan generalisation $f''' + \tfrac{m+1}{2} f f'' + m(1-f'^2) = 0$ with a wedge
parameter $m$ — or make $m$ a network *input* (Example 5's parametric-surrogate trick) to
get all wedge angles in one training.